In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *
import pandas as pd
from show import *

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers.csv', skipinitialspace=True).drop(columns='date').dropna()
dids = df['did'].to_numpy()
xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()

s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xd, yd = s['xd'], s['yd']

s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/residual_20240319.npz')
dx, dy = s['dx'], s['dy']

xd -= dx
yd -= dy

In [3]:
files_fdt = sorted(glob.glob('/home/ulyanov/data/stereo/2025-04-01/fdt/*'))
files_fdt

['/home/ulyanov/data/stereo/2025-04-01/fdt/solo_L2_phi-fdt-bamb_20250401T040003_V202602220258_0544010001.fits.gz',
 '/home/ulyanov/data/stereo/2025-04-01/fdt/solo_L2_phi-fdt-bazi_20250401T040003_V202602220258_0544010001.fits.gz',
 '/home/ulyanov/data/stereo/2025-04-01/fdt/solo_L2_phi-fdt-binc_20250401T040003_V202602220258_0544010001.fits.gz',
 '/home/ulyanov/data/stereo/2025-04-01/fdt/solo_L2_phi-fdt-blos_20250401T040003_V202602220258_0544010001.fits.gz']

In [17]:
file_fdt = files_fdt[3]

with fits.open(file_fdt) as hdul:
    header_fdt = hdul[0].header
    data_fdt = hdul[0].data


did = int(file_fdt.split('_')[-1].split('.')[0])
data_fdt = undistort(data_fdt, header_fdt, xd, yd) * 1.15

view_fdt = View.from_header(header_fdt).update(xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], rsun=ru_sun[dids == did][0],
                                               crota=header_fdt['CROTA'] + 0.25)
view_new = view_fdt.update(nx=1024, ny=1024, rsun=480, xc=511.5, yc=511.5, crota=0, rsun_arc=0)#, crln=view_fdt.crln - 45)

data_fdt = view_new.reproject(data_fdt, view_fdt, mu_thr=0.1)
show_data(data_fdt, view_new, 'FDT', label=r'$B_{los}, G$', vmin=-1000, vmax=1000)
#plt.xlim(320,520)
#plt.ylim(570,770)

In [5]:
files_hmi = sorted(glob.glob('/home/ulyanov/data/stereo/2025-04-01/hmi/*'))
files_hmi

['/home/ulyanov/data/stereo/2025-04-01/hmi/hmi.M_45s.20250401_040600_TAI.2.magnetogram.fits']

In [18]:
file_hmi = files_hmi[0]

with fits.open(file_hmi) as hdul:
    header_hmi = hdul[1].header
    data_hmi = hdul[1].data


data_hmi = rebin(data_hmi, 6, update_header=header_hmi)
view_hmi = View.from_header(header_hmi)

#view_new = view_hmi.update(nx=1024, ny=1024, rsun=480, xc=511.5, yc=511.5, crota=0, rsun_arc=0)
data_hmi = view_new.reproject(data_hmi, view_hmi, mu_thr=0.1)

show_data(data_hmi, view_new, 'HMI', label=r'$B_{los}, G$', vmin=-1000, vmax=1000)
#plt.xlim(320,520)
#plt.ylim(570,770)

In [7]:
transform = view_hmi.to_carrington(origin='heliographic') - view_fdt.to_carrington(origin='heliographic')

e1 = (0,0,1)
e2 = transform(e1)[0]

e1 = np.array(e1)
e2 = np.array(e2)

q = np.sum(e1 * e2)
delta = np.sqrt(1 - q ** 2)

e2_ = (e2 - e1 * q) / delta

v1 = data_fdt.copy()
v2 = data_hmi.copy()

v2[np.abs(v2) < 15] = np.nan

v2_ = (v2 - v1 * q) / delta

print(e2_, np.arccos(q) * 180 / np.pi)

[-0.99504582 -0.09941742  0.        ] 82.68728286948956


In [14]:
with fits.open(files_fdt[1]) as hdul:
    header_azi = hdul[0].header
    data_azi = hdul[0].data

data_azi = view_new.reproject(data_azi, view_fdt)
crota = header_azi['CROTA'] + 0.25
data_azi += crota

In [16]:
amb = (((-np.sin(data_azi * np.pi / 180) * e2_[0] + np.cos(data_azi * np.pi / 180) * e2_[1]) * v2_) < 0)

azi = (data_azi + amb * 180) % 360
azi[np.isnan(v2)] = np.nan

show_data(azi, view_new, title='Stereoscopy', label='Azimuth, degrees', cmap='hsv', vmin=0, vmax=360)
#plt.xlim(320,520)
#plt.ylim(570,770)

In [11]:
with fits.open(files_fdt[1]) as hdul:
    header_azi = hdul[0].header
    data_azi = hdul[0].data

#data_azi = view_new.reproject(data_azi, view_fdt)

with fits.open(files_fdt[0]) as hdul:
    header_amb = hdul[0].header
    data_amb = hdul[0].data[0]

#data_amb = view_new.reproject(data_amb, view_fdt).astype(np.uint32)

In [12]:
def phi_disambig(bazi,bamb,method=2):
    disambig = bamb >> method
    disbazi = bazi.copy()
    disbazi[disambig % 2 != 0] += 180
    return disbazi

In [13]:
azi_ = phi_disambig(data_azi, data_amb)
azi_ += header_azi['CROTA'] + 0.25
azi_ %= 360

azi_ = view_new.reproject(azi_, view_fdt)
azi_[np.isnan(azi)] = np.nan

show_data(azi_, view_new, title='Minimum energy', label='Azimuth, degrees', cmap='hsv', vmin=0, vmax=360)
#plt.xlim(320,520)
#plt.ylim(570,770)

In [124]:
view_fdt.hgln

83.383511

In [123]:
view_hmi.crln

328.869476